# Project 2 — Text-to-Image Search with Pretrained CLIP
Load `openai/clip-vit-base-patch32` and compare natural-language text embeddings with image embeddings. **Inference only; no training.**

In [ ]:
!pip -q install transformers pillow scikit-image matplotlib

## 1. Load pretrained CLIP
First execution downloads model weights; GPU is optional.

In [ ]:
import torch,numpy as np,io,matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel,CLIPProcessor
model_id='openai/clip-vit-base-patch32'
device='cuda' if torch.cuda.is_available() else 'cpu'
model=CLIPModel.from_pretrained(model_id).to(device).eval()
processor=CLIPProcessor.from_pretrained(model_id)
print('Device:',device)

## 2. Create an image gallery
Use scikit-image sample images, or upload your own images in the optional section.

In [ ]:
from skimage import data
samples={'cat':data.chelsea(),'astronaut':data.astronaut(),'coffee':data.coffee(),'rocket':data.rocket(),'horse':data.horse(),'camera':data.camera()}
names=list(samples)
images=[Image.fromarray(np.uint8(arr)).convert('RGB') for arr in samples.values()]
fig,axes=plt.subplots(2,3,figsize=(10,6))
for ax,name,img in zip(axes.flat,names,images):ax.imshow(img);ax.set_title(name);ax.axis('off')
plt.tight_layout();plt.show()

## 3. Encode gallery images once

In [ ]:
with torch.no_grad():
    batch=processor(images=images,return_tensors='pt',padding=True).to(device)
    image_vectors=model.get_image_features(**batch)
    if hasattr(image_vectors,'pooler_output'):image_vectors=image_vectors.pooler_output
    image_vectors=torch.nn.functional.normalize(image_vectors,dim=-1)
print('Image embeddings:',tuple(image_vectors.shape))

## 4. Search images using text and cosine similarity

In [ ]:
def search_images(query,top_k=3):
    with torch.no_grad():
        inputs=processor(text=[query],return_tensors='pt',padding=True,truncation=True).to(device)
        text_vectors=model.get_text_features(**inputs)
        if hasattr(text_vectors,'pooler_output'):text_vectors=text_vectors.pooler_output
        text_vectors=torch.nn.functional.normalize(text_vectors,dim=-1)
        scores=(text_vectors @ image_vectors.T).squeeze(0).cpu().numpy()
    indices=np.argsort(-scores)[:min(top_k,len(images))]
    fig,axes=plt.subplots(1,len(indices),figsize=(4*len(indices),4))
    for ax,i in zip(np.atleast_1d(axes),indices):
        ax.imshow(images[i]);ax.set_title(f'{names[i]}: {scores[i]:.3f}');ax.axis('off')
    plt.suptitle(query);plt.tight_layout();plt.show()
    return [(names[i],float(scores[i])) for i in indices]
search_images('a photograph of a cat')
search_images('a person wearing a space suit')

## 5. Upload your own photos (optional)
Uncomment, run, then rerun Sections 3–4 to rebuild embeddings.

In [ ]:
# from google.colab import files
# uploaded=files.upload()
# images=[Image.open(io.BytesIO(raw)).convert('RGB') for raw in uploaded.values()]
# names=list(uploaded)
# print('Loaded',len(images),'photos. Rerun sections 3 and 4.')

## Student challenges
1. Search for `a small animal` vs `a cat`.
2. Compare `a spacecraft` vs `a rocket`.
3. Upload your own photos and search using words not in the filenames.
4. Why normalize vectors? Why are cosine scores not probabilities?

**Limitations:** Results depend on the image gallery and CLIP's pretrained biases.